# Plasma.jl fit-grid cache

This notebook checks cached and direct WDEM spectra and ForwardDiff gradients. It measures warm objective and gradient times with the same trial points and AtomDB object. Open it with the Julia environment that has this checkout of Plasma.jl and SpectralFitting.jl installed.

Set `atomdb_prefix` to the prefix of your `_line.fits` and `_coco.fits` files, or reuse an existing `db`. If `joint` is already loaded from SpectralGrouping.jl, the objective benchmark uses its Cash or W-stat likelihood and detector responses. Otherwise it uses a synthetic binned spectrum with a chi-squared objective.

In [ ]:
using Plasma, Test, Statistics
import SpectralFitting as SF

atomdb_prefix = joinpath(homedir(), "APED", "apec_v3.1.3")
bench_db = (@isdefined db) ? db : AtomDB(atomdb_prefix)
use_joint = (@isdefined joint)
bench_edges = use_joint ? joint.model_edges :
    reverse(12.39841974 ./ collect(range(1.7, 10.5; length=4401)))
FD = SF.ForwardDiff

function wdem_for_benchmark(db, cache_spectra)
    return Wdem(db; K=1e-3, Tmax=1.2, beta=0.15, inv_slope=0.30,
        abundance=SF.FitParam(0.65; lower_limit=0.01, upper_limit=3.0),
        cache_spectra)
end

bench_cached = wdem_for_benchmark(bench_db, true)
bench_direct = wdem_for_benchmark(bench_db, false)

In [ ]:
@testset "Cached spectra agree with direct synthesis" begin
    for (constructor, settings) in (
            (Apec, (; kT=0.8)),
            (Wdem, (; Tmax=1.2, beta=0.15, inv_slope=0.30)),
            (Bwdem, (; Tmax=1.2, beta=0.15, velocity=200.0)),
            (Cempow, (; Tmax=1.2, alpha=1.1)),
            (LocalCoolflow, (; lowT=0.2, highT=1.2, redshift=-0.005)))
        cached = constructor(bench_db; settings..., K=1e-3)
        direct = constructor(bench_db; settings..., K=1e-3, cache_spectra=false)
        reference = SF.invokemodel(bench_edges, direct)
        SF.invokemodel(bench_edges, cached)
        @test isapprox(SF.invokemodel(bench_edges, cached), reference; rtol=1e-10)
    end
end

In [ ]:
if use_joint
    @eval using SpectralGrouping: grid_statistic, predict_counts
    bench_data = joint
    bench_stat = grid_statistic(joint)
else
    target_model = Wdem(bench_db; K=1.2e-3, Tmax=1.35, beta=0.15,
        inv_slope=0.30, abundance=0.72, cache_spectra=false)
    target = collect(SF.invokemodel(bench_edges, target_model))
    variance = fill((maximum(target) / 50)^2, length(target))
    bench_data = SF.BinnedData(bench_edges, target; codomain_variance=variance)
    bench_stat = SF.ChiSquared()
end

fit_config(model) = SF.FittingConfig(SF.FittingProblem(model => bench_data); stat=bench_stat)
cached_config = fit_config(bench_cached)
direct_config = fit_config(bench_direct)
initial = SF.get_value.(cached_config.u0)

if use_joint && SF.statistic_symbol(bench_stat) in ("C-stat", "W-stat")
    expected_counts = predict_counts(joint, SF.invokemodel(bench_edges, bench_direct))
    actual_counts = only(SF.calculate_objective!(direct_config, initial))
    @test isapprox(actual_counts, expected_counts; rtol=1e-12)
end

trial_points = [initial .* factors for factors in (
    [1.00, 1.00, 1.00, 1.00, 1.00],
    [1.10, 1.07, 1.06, 0.96, 1.05],
    [0.95, 0.92, 0.97, 1.07, 0.90],
)]

make_objective(config) = x -> SF.measure_objective!(config, x, 1)
cached_objective = make_objective(cached_config)
direct_objective = make_objective(direct_config)

@testset "WDEM objective and ForwardDiff gradient" begin
    for point in trial_points
        direct_statistic = direct_objective(point)
        @test isfinite(direct_statistic)
        @test isapprox(cached_objective(point), direct_statistic; rtol=1e-10, atol=1e-10)
        expected = FD.gradient(direct_objective, point)
        actual = FD.gradient(cached_objective, point)
        @test all(isfinite, expected)
        @test isapprox(actual, expected; rtol=1e-8, atol=1e-8)
    end
end

## Warm timings

The agreement checks warm up compilation and load the required AtomDB slices and spectral templates. The table reports median milliseconds per trial, including response folding when using `joint`. Both cases evaluate the same sequence of changing parameters.

In [ ]:
function milliseconds_per_trial(f, points; repeats=7, cycles=5)
    foreach(f, points)
    samples = map(1:repeats) do _
        @elapsed for _ in 1:cycles
            foreach(f, points)
        end
    end
    return 1000 * median(samples) / (cycles * length(points))
end

results = map((
        ("Objective", cached_objective, direct_objective),
        ("ForwardDiff gradient", x -> FD.gradient(cached_objective, x),
            x -> FD.gradient(direct_objective, x)),
    )) do (label, cached, direct)
    cached_ms = milliseconds_per_trial(cached, trial_points)
    direct_ms = milliseconds_per_trial(direct, trial_points)
    (operation=label, cached_ms, direct_ms, speedup=direct_ms / cached_ms)
end
display(collect(results))

## Optional bounded fit

Enable `run_bounded_fit` to compare total fit time with L-BFGS and ForwardDiff. This cell requires Optimization.jl and OptimizationOptimJL.jl. The first solve of each case warms the optimizer. The reported times use the warmed models and keep the same initial values and bounds.

In [ ]:
run_bounded_fit = false

if run_bounded_fit
    @eval import Optimization as Opt
    @eval using OptimizationOptimJL: Optim

    function bounded_fit(objective, config)
        start = SF.get_value.(config.u0)
        lower = SF.get_lowerlimit.(config.u0)
        upper = SF.get_upperlimit.(config.u0)
        scales = max.(abs.(start), 1e-3)
        f = Opt.OptimizationFunction((x, _) -> objective(x .* scales), Opt.AutoForwardDiff())
        problem = Opt.OptimizationProblem(f, start ./ scales; lb=lower ./ scales, ub=upper ./ scales)
        solution = Opt.solve(problem, Optim.Fminbox(Optim.LBFGS()))
        return solution, solution.u .* scales
    end

    for (label, objective, config) in (
            ("cached", cached_objective, cached_config),
            ("direct", direct_objective, direct_config))
        bounded_fit(objective, config)
        result = @timed bounded_fit(objective, config)
        sol, parameters = result.value
        display((case=label, seconds=result.time, status=sol.retcode,
            statistic=sol.objective, parameters))
    end
end